# Yotuubef Colab: Directed Studio Pipeline

This notebook runs the hybrid documentary pipeline in **phases** with a human-in-the-loop checkpoint for script review.

Before running:
1. Set **Runtime -> Change runtime type -> GPU**.
2. Add Colab Secrets (left sidebar key icon): `REDDIT_CLIENT_ID`, `REDDIT_CLIENT_SECRET`, `NVIDIA_NIM_API_KEY`, `YOUTUBE_TOKEN_JSON` (optional).
3. Put one or more `.mp4` background videos in your Drive folder (shown in Cell 2 output).

Workflow:
- **Cell 1-3**: Setup (one-time per session).
- **Cell 4**: Generate script → auto-pauses for your review.
- **Cell 5**: Review and edit the script in an interactive UI.
- **Cell 6**: Render the final video using your edited script.
- **Cell 7**: Download the finished MP4.

In [ ]:
# Cell 1: Environment Setup & Dependencies
import os
import shlex
import subprocess
from pathlib import Path

def run(cmd: str, allow_fail: bool = False) -> None:
    print(f"$ {cmd}")
    completed = subprocess.run(cmd, shell=True, text=True)
    if completed.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed ({completed.returncode}): {cmd}")

# 1. System dependencies
run("apt-get -qq update")
run("apt-get -qq install -y ffmpeg sox git gcc g++")

# 2. Clone repo
REPO_URL = "https://github.com/beenycool/yotuubef.git"
REPO_DIR = Path("/content/yotuubef")

if not REPO_DIR.exists():
    run(f"git clone {shlex.quote(REPO_URL)} {shlex.quote(str(REPO_DIR))}")

os.chdir(REPO_DIR)
run("git pull --ff-only || true", allow_fail=True)

# 3. Python dependencies
run("python3 -m pip install -q --upgrade pip setuptools wheel")
run("python3 -m pip install -q -r requirements-colab.txt")
run("python3 -m pip install -q nest_asyncio ipywidgets")

# 4. Qwen3-TTS (required for voiceover generation)
qwen_cmd = "python3 -m pip install -q git+https://github.com/QwenLM/Qwen3-TTS.git"
print(f"$ {qwen_cmd}")
qwen_res = subprocess.run(qwen_cmd, shell=True, text=True)
if qwen_res.returncode != 0:
    print("Warning: Qwen3-TTS install failed. Pipeline can still run with fallback TTS.")

print("✅ Setup complete.")

In [ ]:
# Cell 2: Mount Drive & Configure Persistence
import os
import shutil
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

REPO_DIR = Path("/content/yotuubef")
os.chdir(REPO_DIR)

DRIVE_ROOT = Path("/content/drive/MyDrive/yotuubef")
PERSIST_ROOT = DRIVE_ROOT / "persistent"
RUNS_ROOT = DRIVE_ROOT / "runs"

# Define persistent directories
important_dirs = {
    "backgrounds": PERSIST_ROOT / "backgrounds",
    "findings": PERSIST_ROOT / "findings",
    "processed": PERSIST_ROOT / "processed",
    "results": PERSIST_ROOT / "results",
    "logs": PERSIST_ROOT / "logs",
    "db": PERSIST_ROOT / "databases",
    "tokens": PERSIST_ROOT / "tokens",
}

for p in [DRIVE_ROOT, PERSIST_ROOT, RUNS_ROOT, *important_dirs.values()]:
    p.mkdir(parents=True, exist_ok=True)

def relink_dir(local_path: Path, drive_path: Path) -> None:
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.is_symlink():
        if local_path.resolve(strict=False) == drive_path.resolve(strict=False):
            return
        local_path.unlink()
    elif local_path.exists():
        if not local_path.is_dir():
            raise RuntimeError(f"Cannot relink non-directory: {local_path}")
        for item in local_path.iterdir():
            target_item = drive_path / item.name
            if not target_item.exists():
                shutil.move(str(item), str(target_item))
        shutil.rmtree(local_path)
    local_path.symlink_to(drive_path, target_is_directory=True)

link_map = {
    REPO_DIR / "findings": important_dirs["findings"],
    REPO_DIR / "processed": important_dirs["processed"],
    REPO_DIR / "logs": important_dirs["logs"],
    REPO_DIR / "data" / "results": important_dirs["results"],
    REPO_DIR / "data" / "logs": important_dirs["logs"],
    REPO_DIR / "data" / "databases": important_dirs["db"],
}

for local_dir, drive_dir in link_map.items():
    relink_dir(local_dir, drive_dir)

print("✅ Drive mounted and persistence linked.")
print(f"📁 Background folder: {important_dirs['backgrounds']}")

In [ ]:
# Cell 3: Secrets & Preflight Check
import json
import os
from pathlib import Path

REPO_DIR = Path("/content/yotuubef")
BACKGROUND_FOLDER_PATH = Path("/content/drive/MyDrive/yotuubef/persistent/backgrounds")
TOKEN_FILE_PATH = Path("/content/drive/MyDrive/yotuubef/persistent/tokens/youtube_token.json")

def get_secret(name: str, default: str = "") -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value if value is not None else default
    except Exception:
        return os.getenv(name, default)

# Map secrets to env vars
secret_map = {
    "REDDIT_CLIENT_ID": "REDDIT_CLIENT_ID",
    "REDDIT_CLIENT_SECRET": "REDDIT_CLIENT_SECRET",
    "NVIDIA_NIM_API_KEY": "NVIDIA_NIM_API_KEY",
    "HACKCLUB_SEARCH_API_KEY": "HACKCLUB_SEARCH_API_KEY",
    "YOUTUBE_TOKEN_JSON": "YOUTUBE_TOKEN_JSON",
}

for secret_name, env_name in secret_map.items():
    val = get_secret(secret_name, "")
    if val:
        os.environ[env_name] = val

os.environ["BACKGROUND_FOLDER"] = str(BACKGROUND_FOLDER_PATH)
os.environ["YOUTUBE_TOKEN_FILE"] = str(TOKEN_FILE_PATH)

youtube_token_json = get_secret("YOUTUBE_TOKEN_JSON", "")
if youtube_token_json:
    try:
        parsed = json.loads(youtube_token_json)
        TOKEN_FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
        TOKEN_FILE_PATH.write_text(json.dumps(parsed), encoding="utf-8")
        print(f"Wrote YouTube token file to: {TOKEN_FILE_PATH}")
    except Exception as exc:
        print(f"Warning: Could not parse/write YOUTUBE_TOKEN_JSON ({exc})")

# --- PREFLIGHT CHECK ---
print("🔍 Running Preflight Checks...")
errors = []

if not os.environ.get("REDDIT_CLIENT_ID"):
    errors.append("❌ REDDIT_CLIENT_ID missing in Colab Secrets")
if not os.environ.get("REDDIT_CLIENT_SECRET"):
    errors.append("❌ REDDIT_CLIENT_SECRET missing in Colab Secrets")
if not os.environ.get("NVIDIA_NIM_API_KEY"):
    errors.append("❌ NVIDIA_NIM_API_KEY missing in Colab Secrets")

bg_videos = list(BACKGROUND_FOLDER_PATH.glob("*.mp4"))
if not bg_videos:
    errors.append(f"❌ No .mp4 background videos found in {BACKGROUND_FOLDER_PATH}")

if errors:
    print("\n".join(errors))
    raise RuntimeError("Preflight checks failed. Please fix the above errors before continuing.")
else:
    print("✅ All API keys present.")
    print(f"✅ Found {len(bg_videos)} background videos.")
    print("✅ Preflight passed. Ready to generate.")

In [ ]:
# Cell 4: Phase 1 - Generate Script (Auto-pauses)
import json
import shlex
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/yotuubef")

# --- CONFIGURATION ---
HYBRID_PROJECT_NAME = "my_documentary_v1"  # Change this per project
HYBRID_REDDIT_URL = ""                     # Optional: e.g., "https://reddit.com/r/..."
HYBRID_NO_UPLOAD = True                    # Keep True for Colab testing

cmd = [
    "python3", "main.py", "hybrid", HYBRID_PROJECT_NAME,
    "--no-upload"
]
if HYBRID_REDDIT_URL.strip():
    cmd += ["--reddit-url", HYBRID_REDDIT_URL.strip()]

cmd_str = " ".join(shlex.quote(part) for part in cmd)
print(f"🚀 Running Phase 1 (Ideas -> Scripting):\n{cmd_str}\n")

def run_pipeline():
    proc = subprocess.Popen(
        cmd, cwd=REPO_DIR, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    return proc.returncode

exit_code = run_pipeline()

# Check if it paused successfully at the review checkpoint
state_path = REPO_DIR / "findings" / HYBRID_PROJECT_NAME / "research" / "state.json"
if state_path.exists():
    state = json.loads(state_path.read_text())
    if state.get("status") == "paused_for_script_review":
        print("\n" + "="*50)
        print("✅ CHECKPOINT REACHED: Script ready for review!")
        print("➡️ Run the next cell to review and edit the script.")
        print("="*50)
elif exit_code != 0:
    print("\n❌ Pipeline failed before reaching checkpoint. Check logs above.")

In [ ]:
# Cell 5: The "Director's Chair" - Review & Edit Script
import json
import ipywidgets as widgets
from IPython.display import display, Markdown
from pathlib import Path

REPO_DIR = Path("/content/yotuubef")
PROJECT_NAME = HYBRID_PROJECT_NAME  # Uses the variable from Cell 4

script_path = REPO_DIR / "findings" / PROJECT_NAME / "research" / "scripts" / "final_script.json"

if not script_path.exists():
    print(f"❌ Script file not found at {script_path}. Did Cell 4 finish successfully?")
else:
    payload = json.loads(script_path.read_text())

    # Create UI Widgets
    title_w = widgets.Text(
        value=payload.get("title", ""),
        description="Title:",
        layout=widgets.Layout(width="90%")
    )
    hook_w = widgets.Textarea(
        value=payload.get("hook", ""),
        description="Hook:",
        layout=widgets.Layout(width="90%", height="60px")
    )

    seg_widgets = []
    for i, seg in enumerate(payload.get("segments", [])):
        w = widgets.Textarea(
            value=seg.get("narration", ""),
            description=f"Seg {i}:",
            layout=widgets.Layout(width="90%", height="60px")
        )
        seg_widgets.append((seg, w))

    save_btn = widgets.Button(description="💾 Save Script Changes", button_style="success")

    def on_save_clicked(b):
        payload["title"] = title_w.value
        payload["hook"] = hook_w.value
        for seg, w in seg_widgets:
            seg["narration"] = w.value
        script_path.write_text(json.dumps(payload, indent=2))
        print("✅ Script saved! You can now run the Render cell.")

    save_btn.on_click(on_save_clicked)

    display(Markdown(f"### 🎬 Reviewing Script: {PROJECT_NAME}"))
    display(Markdown(
        "Edit the text below to improve pacing, fix grammar, or sharpen the hook. "
        "Click save when done."
    ))
    display(title_w)
    display(hook_w)
    for _, w in seg_widgets:
        display(w)
    display(save_btn)

In [ ]:
# Cell 6: Phase 2 - Render Final Video
import shlex
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/yotuubef")
PROJECT_NAME = HYBRID_PROJECT_NAME

cmd = [
    "python3", "main.py", "hybrid", PROJECT_NAME,
    "--resume",
    "--phase", "VIDEO_RENDER",
    "--no-upload"
]

cmd_str = " ".join(shlex.quote(part) for part in cmd)
print(f"🎥 Running Phase 2 (Render Video):\n{cmd_str}\n")

proc = subprocess.Popen(
    cmd, cwd=REPO_DIR, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

if proc.returncode == 0:
    print("\n✅ Render complete!")
else:
    print("\n❌ Render failed. Check logs above.")

In [ ]:
# Cell 7: Download Your Video
from pathlib import Path
from google.colab import files

REPO_DIR = Path("/content/yotuubef")
processed_dir = REPO_DIR / "processed"

videos = sorted(
    processed_dir.glob("*.mp4"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
) if processed_dir.exists() else []

if videos:
    latest = videos[0]
    print(f"📥 Downloading: {latest.name}")
    files.download(str(latest))
else:
    print("❌ No processed .mp4 files found.")